# AgentMorph — Stage 1 Baseline (Colab)

End-to-end Stage 1 workflow on a Colab Pro+ T4. Run cells top-to-bottom.

**Before you start:**
1. **Runtime → Change runtime type → T4 GPU** (the cell below asserts this).
2. Have a HuggingFace token ready with access granted for the gated Meta + Google repos (Llama-3.2-3B-Instruct, Llama-3.1-8B-Instruct, Gemma-2-9b-it).

**Outputs land on Drive:** `/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/`. The runner is resumable via `manifest.json`, so a killed session just means re-running the sweep cell.

## 1. GPU sanity check

In [ ]:
import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)")
else:
    print("NOTE: no CUDA visible. The pipeline will fall back to CPU — "
          "much slower than T4 (~100x) and limited to small models "
          "(Llama-3.2-3B only, likely OOM on 7B+). For production runs "
          "use Runtime > Change runtime type > T4 GPU.")

## 2. Clone / pull the repo

Idempotent: clones on first run, fast-forwards `main` on subsequent runs.

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

> **Security note:** the first Stage 1 run embedded a real HF token in a cell. Treat that token as compromised and rotate it at <https://huggingface.co/settings/tokens>. Then paste the new one below and **clear the cell before sharing the notebook.**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"  # paste, run, then clear this cell
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/AgentMorph[models,smolagents,langgraph,agentdojo]

## 5. Bootstrap (creates cache + runs dir, dry-run smoketest)

In [ ]:
!python /content/AgentMorph/notebooks/colab_setup.py

## 6. (Optional) wipe a prior corrupt run

Only run this if a previous sweep left error-only trajectories (e.g. first run on a CPU runtime or before the smolagents adapter fix). The manifest treats every logged cell as done, so without a wipe the runner won't retry those.

In [ ]:
import shutil, pathlib
RUN_DIR = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline")
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
    print("wiped:", RUN_DIR)
else:
    print("nothing to wipe")

## 7. `native`-framework smoke test

The `native` framework uses our own JSON-in-code-block ReAct loop. It's the simplest path to confirm the model + tools + environment + logger are all healthy. If this finishes cleanly, the rest of the pipeline is sound and remaining bugs are framework-adapter-specific.

In [ ]:
!python -m agentmorph.runner     --model Llama-3.2-3B --framework native --env ecommerce --n-scenarios 5     --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache     --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_smoke

### Inspect the smoke trajectories

In [ ]:
import json, pathlib
smoke_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_smoke/trajectories")
for p in sorted(smoke_dir.glob("*.jsonl")):
    print(f"
=== {p.name} ===")
    for line in p.open():
        t = json.loads(line)
        kinds = [s["kind"] for s in t["steps"]]
        marker = "OK " if t["final_answer"] else "ERR"
        print(f"  [{marker}] {t['scenario_id']:25s} steps={len(t['steps']):2d} kinds={kinds}")

## 8. Full Stage 1 baseline sweep

All 5 models × (smolagents + langgraph) × ecommerce × 20 scenarios = 200 cells. Resumable via `manifest.json`. First run is slow because of HF model downloads (~25 GB, pinned on Drive); resumes are pure inference.

In [ ]:
!python -m agentmorph.runner     --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache     --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_baseline

## 9. Progress peek (run anytime — queues behind the sweep)

In [ ]:
import json, pathlib
manifest = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/manifest.json")
if manifest.exists():
    done = json.loads(manifest.read_text())["completed"]
    print(f"Completed cells: {len(done)} / 200")
    by_model = {}
    for key in done:
        model = key.split("|")[0]
        by_model[model] = by_model.get(model, 0) + 1
    for m, n in sorted(by_model.items()):
        print(f"  {m}: {n}")
else:
    print("Manifest not yet written.")

## 10. Post-run analysis

Run these **after** the sweep finishes.

### Cell A — trajectory counts per (model, framework, env)

In [ ]:
import pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob("*.jsonl")):
    n = sum(1 for _ in p.open())
    print(f"{p.name}: {n} trajectories")

### Cell B — structure of one trajectory

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
sample = next(traj_dir.glob("*.jsonl")).open().readline()
t = json.loads(sample)
print(f"scenario: {t['scenario_id']}")
print(f"model: {t['model_id']}  framework: {t['framework_id']}")
print(f"steps: {len(t['steps'])}  wall_seconds: {t['wall_seconds']:.1f}")
print(f"final_answer: {t['final_answer']!r}
")
for s in t["steps"]:
    tag = s["kind"]
    if tag == "tool_call":
        print(f"  [{tag}] {s['tool_name']}({s['tool_args']})")
    elif tag == "tool_result":
        print(f"  [{tag}] {s['tool_name']} -> {str(s['tool_output'])[:80]}")
    else:
        print(f"  [{tag}] {str(s.get('content',''))[:100]}")

### Cell C — finish-rate table by (model, framework)

In [ ]:
import json, collections, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
stats = collections.defaultdict(lambda: [0, 0])  # (model, framework) -> [total, finished]
for p in traj_dir.glob("*.jsonl"):
    for line in p.open():
        t = json.loads(line)
        key = (t["model_id"], t["framework_id"])
        stats[key][0] += 1
        if t["final_answer"]:
            stats[key][1] += 1
for (m, f), (tot, fin) in sorted(stats.items()):
    pct = 100 * fin / tot if tot else 0
    print(f"{m:18s}  {f:10s}  finished {fin:2d}/{tot:2d}  ({pct:3.0f}%)")